In [2]:
import pandas as pd

DATA_DIR = r"C:\Users\patha\Downloads\data\data"  # adjust if running from a different working directory

golden = pd.read_csv(f"{DATA_DIR}/golden_set_blind.csv")
classified = pd.read_csv(f"{DATA_DIR}/classification_results.csv")
judge = pd.read_csv(f"{DATA_DIR}/judge_scores.csv")
human_judge = pd.read_csv(f"{DATA_DIR}/human_judge_validation.csv")

print(f"Golden set: {len(golden)} rows")
print(f"Classification results: {len(classified)} rows")
print(f"Judge scores: {len(judge)} rows")
print(f"Human judge validation: {len(human_judge)} rows")

Golden set: 187 rows
Classification results: 1200 rows
Judge scores: 40 rows
Human judge validation: 15 rows


In [3]:
def keyword_classify_v2(text):
    """Scoring-based keyword baseline."""
    text_lower = text.lower()
    keyword_sets = {
        "flight_disruption_refund": ["refund", "delayed", "delay", "cancelled", "cancel", "missed connection"],
        "baggage_fees": ["baggage", "checked bag", "carry-on", "carry on", "luggage"],
        "booking_reservation": ["confirmation", "booking", "reservation", "record locator", "itinerary"],
        "technical_app_issue": ["app", "website", "system error", "check-in issue", "login"],
        "service_quality_complaint": ["rude", "terrible", "worst", "poor service", "compensation", "unacceptable", "incompetence"],
        "positive_feedback": ["thank you", "thanks", "love", "amazing", "awesome", "great job"],
        "policy_information": ["policy", "allowed", "how much", "what is the", "rule"],
    }
    scores = {intent: sum(1 for kw in kws if kw in text_lower) for intent, kws in keyword_sets.items()}
    best_intent = max(scores, key=scores.get)
    return best_intent if scores[best_intent] > 0 else "non_actionable_other"

# golden_set_blind.csv intentionally has no predicted_intent (blind labeling) -
# merge it back in from classification_results.csv for the same root_tweet_ids
comparison = golden.merge(classified[["root_tweet_id", "predicted_intent"]], on="root_tweet_id", how="inner")
comparison = comparison.dropna(subset=["human_label", "predicted_intent"]).copy()
comparison["agree"] = comparison["predicted_intent"] == comparison["human_label"]
llm_accuracy = comparison["agree"].mean()

majority_class = comparison["human_label"].value_counts().idxmax()
trivial_accuracy = (comparison["human_label"] == majority_class).mean()

comparison["simple_pred"] = comparison["thread_text"].apply(keyword_classify_v2)
comparison["simple_agree"] = comparison["simple_pred"] == comparison["human_label"]
simple_accuracy = comparison["simple_agree"].mean()

print(f"Trivial (majority class: {majority_class}):  {trivial_accuracy:.1%}")
print(f"Simple (keyword classifier):              {simple_accuracy:.1%}")
print(f"LLM classifier (openai/gpt-oss-20b):      {llm_accuracy:.1%}")

Trivial (majority class: baggage_fees):  18.7%
Simple (keyword classifier):              42.2%
LLM classifier (openai/gpt-oss-20b):      69.0%


In [4]:
per_class = comparison.groupby("predicted_intent")["agree"].agg(["mean", "count"])
per_class.columns = ["accuracy", "n_samples"]
print(per_class.sort_values("accuracy").to_string(float_format=lambda x: f"{x:.1%}"))

                           accuracy  n_samples
predicted_intent                              
non_actionable_other           4.0%         25
policy_information            52.0%         25
service_quality_complaint     64.0%         25
booking_reservation           76.0%         25
technical_app_issue           83.3%         12
positive_feedback             88.0%         25
baggage_fees                  96.0%         25
flight_disruption_refund      96.0%         25


In [5]:
confusion = pd.crosstab(comparison["predicted_intent"], comparison["human_label"], margins=True)
print(confusion.to_string())

human_label                baggage_fees  booking_reservation  flight_disruption_refund  non_actionable_other  policy_information  positive_feedback  service_quality_complaint  technical_app_issue  All
predicted_intent                                                                                                                                                                                        
baggage_fees                         24                    0                         0                     0                   1                  0                          0                    0   25
booking_reservation                   0                   19                         2                     0                   1                  2                          0                    1   25
flight_disruption_refund              0                    0                        24                     0                   0                  0                          1                    0 

In [6]:
merged = human_judge.merge(judge, on="root_tweet_id")
dims = ["groundedness", "relevance", "tone", "actionability"]

print(f"{'Dimension':<16}{'Exact match':>14}{'Within +/-1':>14}{'Mean abs diff':>16}")
for dim in dims:
    diff = (merged[f"human_{dim}"] - merged[dim]).abs()
    print(f"{dim:<16}{diff.eq(0).mean():>13.0%}{diff.le(1).mean():>14.0%}{diff.mean():>16.2f}")

overall = pd.concat([(merged[f"human_{d}"] - merged[d]).abs() <= 1 for d in dims]).mean()
print(f"\nOverall agreement (within +/-1, pooled): {overall:.0%}")

Dimension          Exact match   Within +/-1   Mean abs diff
groundedness              73%           93%            0.33
relevance                 27%           87%            0.93
tone                      40%          100%            0.60
actionability             73%           93%            0.40

Overall agreement (within +/-1, pooled): 93%
